Importing the transformer and specifying the bart language model to be used for the pipeline

In [1]:
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Importing google genai for gemini calls

In [2]:
import google.generativeai as genai
import os

Input Text from the case study

In [3]:
input_text = "Artificial intelligence systems are increasingly embedded in data-driven decision-making across healthcare, finance, transportation, and public services. In hospitals, predictive analytics models are used to estimate patient readmission risks and prioritize clinical attention, while in financial institutions, machine learning systems monitor millions of transactions daily to flag potentially fraudulent behavior. Although these systems promise efficiency and scalability, their effectiveness depends heavily on data quality, model assumptions, and deployment context.\nA recurring challenge is that models trained on historical data may reinforce existing biases rather than eliminate them. For example, a fraud detection model optimized for precision may reduce false positives but simultaneously fail to identify new fraud patterns that deviate from historical trends. Similarly, clinical risk models can perform well on population-level metrics while underperforming for minority groups whose data representation is limited. These limitations are not always apparent to end users, particularly when outputs are summarized or simplified for operational dashboards.\nGenerative AI models are now being integrated to summarize analytical reports, generate explanations, and support conversational interfaces for non-technical stakeholders. However, such models do not possess true understanding and may omit critical caveats, subtly alter numerical relationships, or introduce plausible but incorrect interpretations. As a result, over-reliance on automatically generated summaries or responses can lead to misplaced trust, especially when human oversight is minimal and system outputs are treated as authoritative rather than probabilistic."

Summarized according to minimum and maximum requirements in task 1

In [5]:
result = summarizer(
    input_text,
    max_length=150,
    min_length=120,
    do_sample=False
)


print("Generated Summary: ", result[0]["summary_text"])

Generated Summary:  Artificial intelligence systems are increasingly embedded in data-driven decision-making across healthcare, finance, transportation, and public services. Although these systems promise efficiency and scalability, their effectiveness depends heavily on data quality, model assumptions, and deployment context. A recurring challenge is that models trained on historical data may reinforce existing biases rather than eliminate them. Over-reliance on automatically generated summaries or responses can lead to misplaced trust, especially when human oversight is minimal and system outputs are treated as authoritative rather than probabilistic. It is hoped this article will shed some light on some of the challenges faced by the AI industry.


Got gemini api key from my account and colab user secrets

In [7]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

Checking if the api key is correctly imported and then configuring the gemini api and then specified the Prompt for the context of the usert prompt

In [8]:
if api_key is None:
    raise ValueError("ERROR: GEMINI_API_KEY is not set. Please configure your environment variable or .env file.")

genai.configure(api_key=api_key)

SYSTEM_PROMPT = """
You are an academic-oriented AI assistant.
Follow these rules:
1. Provide clear and factual explanations.
2. Ask for clarification when the user's question is vague.
3. Avoid hallucinations—say “I’m not sure” if information is uncertain.
4. Keep responses concise unless the user requests more detail.
5. Maintain a respectful and helpful tone.
6. Never reveal or guess the API key.
7. Support your answers with step-by-step reasoning whenever useful.
"""

Specified the model for the gemini api as gemini-2.5-flash

In [10]:
model = genai.GenerativeModel("gemini-2.5-flash")


def chat_with_gemini(user_prompt):
    try:
        full_prompt = SYSTEM_PROMPT + "\nUser: " + user_prompt
        response = model.generate_content(full_prompt)
        return response.text.strip()
    except Exception as e:
        return f"[Error: {e}]"


Built the chatbot as an infinite while loop that inputs text and then provides it along with the system prompt to the gemini generate content function to generate a result

In [11]:
print("Welcome to the Gemini Chatbot! Type 'exit' to quit.\n")

while True:
    user_msg = input("You: ").strip()

    if user_msg.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    if user_msg == "":
        print("Chatbot: Please type something.")
        continue

    reply = chat_with_gemini(user_msg)
    print("Chatbot:", reply)


Welcome to the Gemini Chatbot! Type 'exit' to quit.

You: Hello
Chatbot: Hello! How can I assist you today? Please feel free to ask your question.
You: exit
Chatbot: Goodbye!


o	One instance of information loss or distortion observed during summarization

As the total summarization character limits are high so there was no information loss or distortion was observed.
Generated Summary:  Artificial intelligence systems are increasingly embedded in data-driven decision-making across healthcare, finance, transportation, and public services. Although these systems promise efficiency and scalability, their effectiveness depends heavily on data quality, model assumptions, and deployment context. A recurring challenge is that models trained on historical data may reinforce existing biases rather than eliminate them. Over-reliance on automatically generated summaries or responses can lead to misplaced trust, especially when human oversight is minimal and system outputs are treated as authoritative rather than probabilistic. It is hoped this article will shed some light on some of the challenges faced by the AI industry.

o	One quality trade-off introduced by your selected temperature value

Temperature values were not required for the use case as I did the first option and not the second one.

o	One limitation of rule-based control in chatbot responses

People might be able to exploit vulnerabilities if certain context is not directly specified such as asking my collab notebook information as I did not restrict that.
